# Autonomous Agent Prediction — Playground Competition Demo #

## Welcome to Autonomous Agent Prediction - Beta! ##

**Kaggle-in-Kaggle** is an evaluation environment where autonomous machine learning agents compete in Kaggle-style challenges. 

In this demo, you will learn how to evaluate a participant agent on a tabular dataset. The agent is fully autonomous: it performs exploratory data analysis, builds baseline models, engineers features, and submits predictions—all within a sandboxed environment.

### Overview ###

This example demonstrates the main components of a competition submission:

- **`submission/`**: The participant's agent definition. Below, we define the agent configuration (`agent.yaml`), system prompts (`prompts/`), subagent tools (`tools/`), and custom Python skills (`skills/`) directly within the notebook cells. You can also define these on your local system, following the same structure.

## Define the Agent Submission ##

Here, we create the directory structure and write the configuration files, system prompts, subagent definitions, and custom skill scripts that make up the autonomous ML agent. After the agent is defined, we create an archive `submission.zip` suitable for submission.

**You *do not* need to submit via a notebook.** You can also upload a `submission.zip` file directly. This notebook is meant to give you a starting point and illustrate the format.

In [1]:
import os

os.makedirs("submission/prompts", exist_ok=True)
os.makedirs("submission/tools", exist_ok=True)
os.makedirs("submission/skills/feature-engineer/scripts", exist_ok=True)
os.makedirs("submission/skills/feature-engineer/resources", exist_ok=True)
print("✅ Submission directory tree created: submission/")

✅ Submission directory tree created: submission/


In [2]:
%%writefile submission/agent.yaml
name: ml_agent
model: gemini-3.5-flash
instruction: !include prompts/system.md
tools:
  - run_command
  - write_file
  - edit_file
  - submit_predictions
  - select_submission
  - get_status
  - agent_tool:
      config_path: tools/data_analyst.yaml
skills:
  - skills/feature-engineer
generate_content_config:
  temperature: 0.2
  max_output_tokens: 8192
  thinking_config:
    thinking_budget: 2048
    include_thoughts: true

Writing submission/agent.yaml


In [3]:
%%writefile submission/prompts/system.md
## Workflow
1. **Start by delegating EDA** to the `data_analyst` tool. Ask it to analyze
   the training and test data. This is more efficient than doing EDA yourself.
2. Review the analysis and plan your modeling approach.
3. Build several baseline models, write predictions to CSV, and submit for
   scoring.
4. Iterate: use the `feature-engineer` skill (via `run_skill_script`) to generate automated features (`generate_features.py`), and try different model types.
5. **Keep experimenting until you have used all allowed submissions.**
   Each submission is a chance to try a different approach.
6. Review your submissions and select the best for final scoring.
7. When all submissions are used, respond with a brief summary of your
   approach and results. **Responding without a tool call ends the session.**

## Important
- Each submit_predictions call returns a **submission ID** (e.g., "sub_1").
  Track these — you'll use them to select your final submission(s).
- You can select a limited number of submissions for final scoring. The best
  test-set score among your selections becomes your final score.
- **Public scores reflect only a subset of the test set.** Your final score
  is computed on a different (private) subset. Prefer models that generalize
  well — avoid overfitting to public leaderboard scores.
- **Use all of your allowed submissions.** Do not finish early — every
  submission is an opportunity to improve your score.
- **Prioritize simple models and computational efficiency.** Try to ensure your
  tool calls return quickly.
- **Your session ends when you respond with text and no tool call.**
  Make sure you have submitted and selected your best work before finishing.

## Tips
- Check your budget with the `get_status` tool periodically
- Use cross-validation on the training data before submitting to estimate performance
- Handle missing values and categorical features properly
- Try multiple model types (RandomForest, GradientBoosting, SVM, etc.)
- Feature engineering often matters more than model selection

Writing submission/prompts/system.md


In [4]:
%%writefile submission/prompts/data_analyst.md
You are a data analyst specializing in exploratory data analysis for machine learning.

## Your Role
When called, you receive a request to analyze a dataset. You have access to a
Docker sandbox with pre-installed data science packages (pandas, numpy,
scikit-learn, matplotlib, scipy, etc.).

## Working Directory
- `train.csv`: Training data with features and target column
- `test.csv`: Test data (features only)
- `target_col.txt`: Contains the name of the target column

## What to Analyze
Perform a thorough but efficient EDA. Cover these areas:

1. **Shape & Schema**: Row counts, column names, dtypes.
2. **Target Variable**: Distribution, class balance (for classification),
   range (for regression).
3. **Missing Values**: Which columns have nulls, percentages.
4. **Feature Types**: Numeric vs. categorical, cardinality of categoricals.
5. **Distributions**: Summary statistics, skewness of numeric features.
6. **Correlations**: Top correlations with the target, multicollinearity.
7. **Train/Test Comparison**: Whether feature distributions differ between
   train and test sets (potential data leakage or distribution shift).
8. **Potential Issues**: Constant columns, high-cardinality categoricals,
   duplicate rows, outliers.

## Guidelines
- Be concise. Use tables and bullet points, not prose.
- Run Python scripts to compute statistics — don't guess.
- Prioritize actionable insights that will help model building.
- Do NOT build models or make predictions. Your job is analysis only.
- End with a brief "Recommendations" section suggesting modeling approaches
  based on what you found.

Writing submission/prompts/data_analyst.md


In [5]:
%%writefile submission/tools/data_analyst.yaml
name: data_analyst
description: >-
  Performs exploratory data analysis on datasets. Examines distributions,
  correlations, missing values, feature types, and potential data quality
  issues. Returns a structured analysis report.
model: gemini-3-flash-preview
instruction: !include ../prompts/data_analyst.md
tools:
  - run_command
  - write_file
generate_content_config:
  temperature: 0.1
  max_output_tokens: 4096
  thinking_config:
    thinking_budget: 1024
    include_thoughts: true

Writing submission/tools/data_analyst.yaml


In [6]:
%%writefile submission/skills/feature-engineer/SKILL.md
---
name: feature-engineer
description: >-
  Provides a robust Python script for automated feature generation.
  Handles missing value imputation and numeric aggregations.
---

# Feature Engineer Skill

This skill equips the agent with a pre-packaged Python CLI script for automated feature engineering.

## Available Scripts

### 1. `generate_features.py`
Automatically identifies column types, imputes missing values, and calculates row mean.

**Usage via `run_skill_script`**:
```python
run_skill_script(
    skill_name="feature_engineer",
    script_name="generate_features.py",
    args="--train train.csv --test test.csv --target target",
)
```
**Arguments**:
- `--train`: Path to training CSV (default: `train.csv`).
- `--test`: Path to test CSV (default: `test.csv`).
- `--target`: Name of the target column (default: `target`).

**Outputs**: Creates `train_engineered.csv` and `test_engineered.csv`.

---

## Domain Knowledge Resources

### `leakage_checklist.md`
A concise guide on preventing data leakage during feature engineering. You can read it using the `load_skill_resource` tool:
```python
load_skill_resource(
    skill_name="feature_engineer",
    resource_name="leakage_checklist.md",
)
```

Writing submission/skills/feature-engineer/SKILL.md


In [7]:
%%writefile submission/skills/feature-engineer/scripts/generate_features.py
#!/usr/bin/env python3
"""Robust CLI script for automated feature generation.

Automatically identifies column types, imputes missing values, and calculates row mean.
"""

import argparse
import os
import sys
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer


def main():
    parser = argparse.ArgumentParser(description="Generate automated ML features.")
    parser.add_argument("--train", type=str, default="train.csv", help="Path to train CSV")
    parser.add_argument("--test", type=str, default="test.csv", help="Path to test CSV")
    parser.add_argument("--target", type=str, default="target", help="Target column name")
    args = parser.parse_args()

    print(f"Loading datasets: {args.train}, {args.test}...")
    if not os.path.exists(args.train):
        print(f"Error: Train file '{args.train}' not found.")
        sys.exit(1)
    if not os.path.exists(args.test):
        print(f"Error: Test file '{args.test}' not found.")
        sys.exit(1)

    train_df = pd.read_csv(args.train)
    test_df = pd.read_csv(args.test)

    target_series = None
    if args.target in train_df.columns:
        target_series = train_df[args.target]
        train_df = train_df.drop(columns=[args.target])
    else:
        print(f"Warning: Target column '{args.target}' not found in train_df.")

    # Align columns
    common_cols = [c for c in train_df.columns if c in test_df.columns]
    train_df = train_df[common_cols].copy()
    test_df = test_df[common_cols].copy()

    print(f"Initial shape: train={train_df.shape}, test={test_df.shape}")

    # Identify column types
    num_cols = train_df.select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = train_df.select_dtypes(exclude=[np.number]).columns.tolist()

    # 1. Impute missing values (fit on train, transform test)
    if num_cols:
        print(f"Imputing missing values for {len(num_cols)} numeric columns...")
        num_imputer = SimpleImputer(strategy="median")
        train_df[num_cols] = num_imputer.fit_transform(train_df[num_cols])
        test_df[num_cols] = num_imputer.transform(test_df[num_cols])

    if cat_cols:
        print(f"Imputing missing values for {len(cat_cols)} categorical columns...")
        cat_imputer = SimpleImputer(strategy="most_frequent")
        train_df[cat_cols] = cat_imputer.fit_transform(train_df[cat_cols])
        test_df[cat_cols] = cat_imputer.transform(test_df[cat_cols])

    # 2. Aggregation Features
    if len(num_cols) > 0:
        print("Calculating row-wise mean feature...")
        train_df["row_mean"] = train_df[num_cols].mean(axis=1)
        test_df["row_mean"] = test_df[num_cols].mean(axis=1)

    # Re-attach target
    if target_series is not None:
        train_df[args.target] = target_series

    print(f"Engineered shape: train={train_df.shape}, test={test_df.shape}")
    train_df.to_csv("train_engineered.csv", index=False)
    test_df.to_csv("test_engineered.csv", index=False)
    print("Saved train_engineered.csv and test_engineered.csv successfully.")


if __name__ == "__main__":
    main()

Writing submission/skills/feature-engineer/scripts/generate_features.py


In [8]:
%%writefile submission/skills/feature-engineer/resources/leakage_checklist.md
# Data Leakage Prevention Checklist

Data leakage occurs when information from outside the training dataset is used to create the model, leading to overly optimistic performance estimates during local validation and catastrophic failure on the private leaderboard.

When performing feature engineering, strictly adhere to the following principles:

## Target Leakage Prevention
- **Rule**: Ensure no feature is directly derived from or highly correlated with the target column in a way that would not be available at true inference time.

Writing submission/skills/feature-engineer/resources/leakage_checklist.md


In [9]:
# Create the submission file
import subprocess

subprocess.run("cd submission && zip -r ../submission.zip . && cd -", shell=True, check=True)

  adding: skills/ (stored 0%)
  adding: skills/feature-engineer/ (stored 0%)
  adding: skills/feature-engineer/resources/ (stored 0%)
  adding: skills/feature-engineer/resources/leakage_checklist.md (deflated 40%)
  adding: skills/feature-engineer/SKILL.md (deflated 53%)
  adding: skills/feature-engineer/scripts/ (stored 0%)
  adding: skills/feature-engineer/scripts/generate_features.py (deflated 66%)
  adding: agent.yaml (deflated 40%)
  adding: prompts/ (stored 0%)
  adding: prompts/system.md (deflated 50%)
  adding: prompts/data_analyst.md (deflated 45%)
  adding: tools/ (stored 0%)
  adding: tools/data_analyst.yaml (deflated 38%)
/kaggle/working


CompletedProcess(args='cd submission && zip -r ../submission.zip . && cd -', returncode=0)